# E-commerce Bronze → Silver / Rejected
Uses PostgreSQL structural schema snapshot and `etl.yaml` business rules. Attach **lh_ecommerce** as default Lakehouse. Upload `etl.yaml` and `schema_metadata.json` into `Files/config/`.

In [ ]:
import json, yaml, re
from functools import reduce
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType

BASE = "/lakehouse/default/Files/config"
with open(f"{BASE}/etl.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
with open(f"{BASE}/schema_metadata.json", encoding="utf-8") as f:
    meta = json.load(f)
DATASETS = cfg['load_order']
assert set(DATASETS) == set(cfg['tables']) == set(meta)
print('Configuration loaded:', DATASETS)


## Strict casting (reject invalid casts rather than silently producing NULL)

In [ ]:
def spark_type(pg_type):
    p = pg_type.upper().strip()
    if re.match(r'^(BIGINT|BIGSERIAL|INT8)\b', p): return 'bigint'
    if re.match(r'^(SMALLINT|SMALLSERIAL|INT2)\b', p): return 'smallint'
    if re.match(r'^(INTEGER|INT|SERIAL|INT4)\b', p): return 'int'
    if re.match(r'^(NUMERIC|DECIMAL)\b', p):
        match = re.search(r'\((\d+)\s*,\s*(\d+)\)',p)
        return f'decimal({match.group(1)},{match.group(2)})' if match else 'decimal(38,10)'
    if re.match(r'^(DOUBLE PRECISION|FLOAT8|REAL|FLOAT4|FLOAT)\b',p): return 'double'
    if re.match(r'^BOOLEAN\b',p): return 'boolean'
    if re.match(r'^TIMESTAMP',p): return 'timestamp'
    if re.match(r'^DATE\b',p): return 'date'
    if re.match(r'^(CHARACTER VARYING|VARCHAR|CHAR|CHARACTER|TEXT)\b',p): return 'string'
    raise ValueError(f'Unsupported PostgreSQL type: {pg_type}')

# Fail fast if unsupported physical schema is found.
for name in DATASETS:
    for field, details in meta[name]['columns'].items():
        spark_type(details['type'])

def reject(current, rejected, condition, reason):
    bad = current.filter(F.coalesce(condition, F.lit(False)))
    rejected.append(bad.withColumn('reject_reason', F.lit(reason)))
    good = current.filter(~F.coalesce(condition, F.lit(False)))
    return good

def validate_structural(df, spec):
    rejected = []
    columns = spec['columns']
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f'Missing Bronze columns: {sorted(missing)}')
    extra = set(df.columns) - set(columns)
    if extra:
        raise ValueError(f'Unexpected Bronze columns: {sorted(extra)}')
    current = df
    for colname, details in columns.items():
        target = spark_type(details['type'])
        if target != 'string':
            converted = F.expr(f'try_cast(`{colname}` as {target})')
            current = current.withColumn('__cast_value', converted)
            current = reject(current, rejected, F.col(colname).isNotNull() & F.col('__cast_value').isNull(), f'INVALID_DATATYPE_{colname.upper()}')
            current = current.withColumn(colname, F.col('__cast_value')).drop('__cast_value')
    for colname, details in columns.items():
        if not details['nullable']:
            current = reject(current, rejected, F.col(colname).isNull(), f'NULL_{colname.upper()}')
    for colname, details in columns.items():
        mt = re.match(r'^(?:CHARACTER VARYING|VARCHAR|CHARACTER|CHAR)\s*\((\d+)\)',details['type'].upper())
        if mt:
            current = reject(current, rejected, F.col(colname).isNotNull() & (F.length(F.col(colname)) > int(mt.group(1))), f'MAX_LENGTH_{colname.upper()}')
    seen = set()
    for keys in [spec.get('primary_key', [])] + spec.get('unique_constraints',[]):
        keys = tuple(keys)
        if not keys or keys in seen: continue
        seen.add(keys)
        # Keep one deterministic representative per key. Identical duplicate copies are interchangeable.
        stable_sort = F.sha2(F.to_json(F.struct(*[F.col(c) for c in sorted(columns)]), options={'ignoreNullFields':'false'}),256)
        w = Window.partitionBy(*keys).orderBy(stable_sort)
        current = current.withColumn('__row_number', F.row_number().over(w))
        current = reject(current, rejected, F.col('__row_number') > 1, 'DUPLICATE_'+'_'.join(keys).upper()).drop('__row_number')
        rejected[:] = [part.drop('__row_number') if '__row_number' in part.columns else part for part in rejected]
    return current, rejected


## Foreign keys and business rules

In [ ]:
def validate_fk(current, rejected, spec, clean):
    for fk in spec.get('foreign_keys', []):
        keys, refs = fk['columns'], fk['reference_columns']
        parent = clean[fk['reference_dataset']]
        marker = parent.select(*[F.col(r).alias(f'__fk_{i}') for i,r in enumerate(refs)]).distinct().withColumn('__exists',F.lit(1))
        cond = reduce(lambda a,b:a&b,[F.col(f'l.`{k}`') == F.col(f'r.__fk_{i}') for i,k in enumerate(keys)])
        joined = current.alias('l').join(marker.alias('r'),cond,'left')
        nullkey = reduce(lambda a,b:a|b,[F.col(f'l.`{k}`').isNull() for k in keys])
        bad = joined.filter(~nullkey & F.col('r.__exists').isNull()).select('l.*')
        rejected.append(bad.withColumn('reject_reason',F.lit('INVALID_FK_'+'_'.join(keys).upper())))
        current = joined.filter(nullkey | F.col('r.__exists').isNotNull()).select('l.*')
    return current

def validate_business(name, current, rejected, clean):
    rules = cfg.get('datasets', {}).get(name, {}).get('rules', [])
    for rule in rules:
        kind, colname = rule['type'], rule['column']
        if kind == 'allowed_values':
            invalid = F.col(colname).isNotNull() & ~F.col(colname).isin(rule['values'])
        elif kind == 'min_value':
            invalid = F.col(colname).isNotNull() & ((F.col(colname) < F.lit(rule['value'])) if rule.get('inclusive', True) else (F.col(colname) <= F.lit(rule['value'])))
        elif kind == 'equals_reference':
            ref = clean[rule['reference_dataset']].select(F.col(rule['join_on']).alias('__join_key'),F.col(rule['reference_column']).alias('__reference_value'))
            current = current.join(ref,current[rule['join_on']]==ref['__join_key'],'left').drop('__join_key')
            invalid = F.col('__reference_value').isNotNull() & (F.abs(F.col(colname)-F.col('__reference_value')) > F.lit(rule.get('tolerance',0)))
        else:
            raise ValueError(f'Unsupported business rule: {kind}')
        current = reject(current,rejected,invalid,rule['reason'])
        if kind == 'equals_reference':
            current = current.drop('__reference_value')
            rejected[:] = [v.drop('__reference_value') if '__reference_value' in v.columns else v for v in rejected]
    return current


## Process all datasets; verify row-level partition counts; only then write Delta tables

In [ ]:
clean = {}
rejected = {}
summary = []
outputs = {}
for name in DATASETS:
    source = spark.table(f'bronze_{name}')
    raw_count = source.count()
    valid, parts = validate_structural(source, meta[name])
    valid = validate_fk(valid,parts,meta[name],clean)
    valid = validate_business(name,valid,parts,clean)
    if parts:
        bad = reduce(lambda a,b:a.unionByName(b,allowMissingColumns=True),parts)
    else:
        bad = source.limit(0).withColumn('reject_reason',F.lit(None).cast('string'))
    clean_count, bad_count = valid.count(),bad.count()
    assert raw_count == clean_count + bad_count, f'{name}: reconciliation FAIL ({raw_count} != {clean_count}+{bad_count})'
    # Cache because child FK checks depend on clean parents.
    valid = valid.cache()
    valid.count()
    clean[name] = valid
    rejected[name] = bad
    outputs[name] = (valid,bad)
    summary.append((name,raw_count,clean_count,bad_count))
    print(name, 'raw=',raw_count, 'clean=',clean_count, 'rejected=',bad_count)

# IMPORTANT: This is row-count reconciliation, not an independent proof that rejects are correct.
# Stop immediately if any dataset fails counts or schema checks.
for name,(good,bad) in outputs.items():
    good.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'silver_{name}')
    bad.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'rejected_{name}')

summary_df = spark.createDataFrame(summary,['dataset','raw_rows','silver_rows','rejected_rows'])
display(summary_df)


## Inspect reject reasons and silver output

In [ ]:
for name in DATASETS:
    print(f'== {name} ==')
    display(spark.table(f'rejected_{name}').groupBy('reject_reason').count().orderBy('reject_reason'))
# Example:
display(spark.table('silver_customers').limit(10))


## Limitations and follow-up tests
- This notebook intentionally follows the existing Python transform: the **first failing rule wins**, so a row is rejected once.
- It does **not** independently verify reject correctness. Add a separate expected-rule oracle and multiset comparison tests.
- If CSV ingestion already converted empty strings to null, Bronze cannot recover their original representation; use source-file checks.
- Two-phase validation happens before output overwrite, but Spark does not provide an atomic transaction across all ten output tables.
- `ROW_NUMBER` selects a deterministic representative by row-content hash; for identical copies, the retained instance is interchangeable.
- The code supports basic PostgreSQL types; unknown types intentionally fail fast.